# Step 3 — Visualize EWCE Disease Genes Across Brain Regions

**Kernel:** `sc_gpu` conda env

Generates EWCE overview heatmap, dot plot, and per-vascular-cell-type pseudobulk
heatmaps (region × gene). Optionally calls R for publication-quality ComplexHeatmap PDFs.

**Outputs → `../Results/`:**
`ewce_overview_all_celltypes.svg`, `ewce_dotplot_all_celltypes.svg`,
`heatmap_pseudobulk_{ct}.csv`, `heatmap_{ct}.svg`, `heatmap_{ct}_complexheatmap.pdf`

## 1. Imports & Gene List

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — edit the values in this cell, then run all cells
# ═══════════════════════════════════════════════════════════════════════════

# ── Disease gene list CSV ───────────────────────────────────────────────────
GENE_CSV_PATH   = "/home/kibr/Work/Vas_try/Data/brain_vascular_disease_genes_curated.csv"   # ← set your path
DISEASE_COL     = "condition"     # ← column name for disease labels
GENE_COL        = "gene" #"gene_name"  # ← column name for gene symbols

# ── H5AD single-cell atlas ──────────────────────────────────────────────────
H5AD_PATH       = "/home/kibr/Work/Vascular_atlas/Results/Revision_2/clean_object_revision_03032026.h5ad"              # ← set your path
CELL_CLASS_COL  = "Cell_class"  # ← obs column for cell type labels
DONOR_COL       = "individualID"# ← obs column for donor / sample ID

# ── Output directory ────────────────────────────────────────────────────────
OUT             = "/home/kibr/Work/Vas_try/Results"              # ← set your path

# ── Analysis parameters (safe to leave as-is) ──────────────────────────────
N_BOOT          = 10_000   # bootstrap iterations
SEED            = 42       # random seed
CHUNK           = 2000     # gene-chunk size for CPM computation

# ── Vascular cell types to highlight (adjust to match your atlas) ──────────
VASC_TYPES      = ["Endothelial", "Mural_Cell", "Fibroblast"]

print("Configuration loaded.")
print(f"  Gene CSV  : {GENE_CSV_PATH}")
print(f"  H5AD      : {H5AD_PATH}")
print(f"  Output dir: {OUT}")


In [ ]:
%matplotlib inline
import numpy as np, scipy.sparse as sp, anndata as ad, os, time
from collections import defaultdict
from scipy.stats import false_discovery_control
import csv, pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import matplotlib.cm as cm
import scanpy as sc
import pandas as pd
from scipy import sparse
import seaborn as sns

# ── Load gene list using user-specified columns ──────────────────────────────
df_genes = pd.read_csv(GENE_CSV_PATH)

# Validate required columns
for col in [DISEASE_COL, GENE_COL]:
    if col not in df_genes.columns:
        raise ValueError(f"Column '{col}' not found in CSV. "
                         f"Available columns: {df_genes.columns.tolist()}")

# Normalise column names for downstream use
if DISEASE_COL != 'disease' or GENE_COL != 'gene_name':
    df_genes = df_genes.rename(columns={DISEASE_COL: 'disease', GENE_COL: 'gene_name'})

print(df_genes.columns)
print(f"Loaded {len(df_genes)} entries | {df_genes['gene_name'].nunique()} unique genes")

display(df_genes.groupby('disease')['gene_name'].count().rename('n_genes').to_frame())


In [ ]:
# %matplotlib inline
# import numpy as np, scipy.sparse as sp, anndata as ad, os, subprocess, textwrap
# import seaborn as sns, matplotlib.patches as mpatches
# import matplotlib.pyplot as plt, matplotlib.patches as mpatches
# plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})
# import csv, pandas as pd, numpy as np
# import matplotlib.pyplot as plt, matplotlib.patches as mpatches
# import matplotlib.cm as cm

# def load_gene_list(path):
#     """
#     Read brain_vascular_disease_genes_v3.csv robustly.
#     Fixes column-shift caused by unquoted commas in Evidence_Level.
#     Returns DataFrame with columns: disease, gene_name, pathway, pathway_short,
#                                     Susceptible_Cell_Types, Gene_Type, Evidence_Level
#     """
#     def fix_row(row, n=8):
#         # Real pathways always contain '/' — merge fragment back into Evidence_Level
#         while len(row) > 5:
#             cand = row[5] if len(row) > 5 else ''
#             if '/' in cand or not cand.strip():
#                 break
#             row[4] = row[4] + ',' + row.pop(5)
#         return row[:n]

#     rows = []
#     with open(path) as f:
#         reader = csv.reader(f)
#         header = next(reader)[:8]
#         for row in reader:
#             rows.append(fix_row(row))

#     df = pd.DataFrame(rows, columns=header)
#     df.columns = ['disease', 'Stroke_Subtype', 'gene_name', 'Gene_Type',
#                   'Evidence_Level', 'pathway', 'Susceptible_Cell_Types', 'Notes']
#     df = df[df['gene_name'].str.strip().ne('')].copy()
#     # Short pathway label: first segment before '/'
#     df['pathway_short'] = df['pathway'].str.split('/').str[0].str.strip()
#     df['pathway_short'] = df['pathway_short'].replace('', 'Other')
#     df.loc[df['pathway'].str.strip().eq(''), 'pathway_short'] = 'Other'
#     return df

# GENE_CSV_PATH = "/home/kibr/Work/Vas_try/Data/brain_vascular_disease_genes_curated.csv"
# df_genes = load_gene_list(GENE_CSV_PATH)
# print(f"Loaded {len(df_genes)} entries | {df_genes['gene_name'].nunique()} unique genes | "
#       f"{df_genes['disease'].nunique()} diseases | {df_genes['pathway_short'].nunique()} pathway groups")
# display(df_genes.groupby('disease')['gene_name'].count().rename('n_genes').to_frame())

## 2. Colour Palettes

In [ ]:
# ── Auto-generate disease & pathway colour palettes ─────────────────────────
diseases = sorted(df_genes['disease'].unique())

_dis_cmap = plt.get_cmap('tab20', len(diseases))
DISEASE_COLORS = {
    d: '#{:02X}{:02X}{:02X}'.format(*[int(x * 255) for x in _dis_cmap(i)[:3]])
    for i, d in enumerate(diseases)
}

# Keep vascular type colours fixed
VASC_COLORS = {"Endothelial": "#fcbe05", "Mural_Cell": "#A26DC2", "Fibroblast": "#5b844d"}
CT_COLORS = {
    "Astrocyte": "#F06719", "Neuron": "#08415C", "Ependymal_Cell": "#23767C",
    "Epithelial_Cell": "#155a66", "Microglia_Macrophage_T": "#DC143C",
    "Oligodendrocyte": "#00BFC4", "Endothelial": "#fcbe05",
    "Mural_Cell": "#A26DC2", "Fibroblast": "#5b844d", "OPC": "#0072B2",
}

# Preview colour strips
fig, axes = plt.subplots(1, 1, figsize=(14, 3))
for ax, pal, title in [(axes, DISEASE_COLORS, 'Disease colours')]:
    for i, (lbl, col) in enumerate(pal.items()):
        ax.barh(0, 1, left=i, color=col, edgecolor='white', linewidth=0.5)
        ax.text(i + 0.5, 0, lbl[:18], ha='center', va='center',
                fontsize=5, rotation=90, color='white')
    ax.set_xlim(0, len(pal))
    ax.axis('off')
    ax.set_title(title, fontsize=9, loc='left')

plt.tight_layout()
plt.show()

## 3. Configuration

In [ ]:
DATA      = "/home/kibr/Work/Vascular_atlas/Results/Revision_2/clean_object_revision_03032026.h5ad"
TOP_GENES = "/home/kibr/Work/Vas_try/Results/ewce_top_genes.csv"
OUT       = "/home/kibr/Work/Vas_try/Results"
RSCRIPT   = "/home/kibr/Softwares/miniforge3/envs/sc_r/bin/Rscript"
MIN_CELLS = 50
VASC_TYPES = ["Endothelial", "Mural_Cell", "Fibroblast"]
os.makedirs(OUT, exist_ok=True)
print("Config OK.")

## 4. Load AnnData + Library Sizes


In [ ]:
print("Loading AnnData (backed mode) ...")
adata     = ad.read_h5ad(DATA, backed='r')
obs       = adata.obs[["Cell_class", "brain_region", "individualID"]].copy()
var_names = adata.var_names.tolist()
gene2idx  = {g: i for i, g in enumerate(var_names)}
n_genes   = adata.n_vars
print(f"  shape={adata.shape}")

In [ ]:
print("Pre-computing library sizes ...")
LCHUNK    = 3000
lib_sizes = np.zeros(adata.n_obs, dtype=np.float64)
for gs in range(0, n_genes, LCHUNK):
    ge  = min(gs + LCHUNK, n_genes)
    blk = adata.X[:, gs:ge]
    lib_sizes += np.asarray(blk.sum(axis=1) if sp.issparse(blk)
                            else blk.sum(axis=1)).flatten()
    if gs % 15000 == 0:
        print(f"  {gs}/{n_genes}", flush=True)
lib_sizes = np.maximum(lib_sizes, 1)
print(f"Done. min={lib_sizes.min():.0f}  max={lib_sizes.max():.0f}")

## 5. Load EWCE Top Genes

In [ ]:
df_top = pd.read_csv(TOP_GENES)
# Merge pathway_short from df_genes if not already present
# if 'pathway_short' not in df_top.columns:
#     pw_map = df_genes.drop_duplicates('gene_name').set_index('gene_name')['pathway_short']
#     df_top['pathway_short'] = df_top['gene_name'].map(pw_map).fillna('Other')

print(f"Top genes: {len(df_top)} rows | {df_top['cell_type'].nunique()} cell types")
display(df_top.groupby("cell_type")[["gene_name","rank","disease","specificity_score"]]
        .apply(lambda x: x.sort_values("rank")).reset_index(drop=True))

## 6. Helper Functions

In [ ]:
def compute_pseudobulk(adata, obs, cell_type, genes, min_cells=MIN_CELLS):
    mask_ct   = (obs["Cell_class"] == cell_type).values
    obs_ct    = obs[mask_ct]
    regions   = obs_ct["brain_region"].value_counts()
    regions   = regions[regions >= min_cells].index.tolist()
    genes_in  = [g for g in genes if g in gene2idx]
    g_idxs    = [gene2idx[g] for g in genes_in]
    cell_idxs = np.where(mask_ct)[0]
    rows, labels = [], []
    for reg in regions:
        reg_mask      = (obs_ct["brain_region"] == reg).values
        reg_cell_idxs = cell_idxs[reg_mask]
        block = adata.X[reg_cell_idxs, :][:, g_idxs]
        if sp.issparse(block): block = block.toarray()
        block  = block.astype(np.float64)
        cpm    = block / lib_sizes[reg_cell_idxs, None] * 1e6
        rows.append(np.log2(cpm + 1).mean(axis=0))
        labels.append(reg)
    if not rows: return pd.DataFrame(), pd.DataFrame()
    df_lc = pd.DataFrame(np.vstack(rows), index=labels, columns=genes_in)
    df_z  = (df_lc - df_lc.mean()) / df_lc.std().replace(0, 1)
    return df_z, df_lc

def compute_dotplot_stats(adata, obs, genes, cell_types):
    genes_in = [g for g in genes if g in gene2idx]
    g_idxs   = [gene2idx[g] for g in genes_in]
    records  = []
    for ct in cell_types:
        idxs  = np.where((obs["Cell_class"] == ct).values)[0]
        block = adata.X[idxs, :][:, g_idxs]
        if sp.issparse(block): block = block.toarray()
        block   = block.astype(np.float32)
        ct_lib  = lib_sizes[idxs].astype(np.float32)[:, None]
        lognorm = np.log2(block / ct_lib * 1e4 + 1)
        for gi, g in enumerate(genes_in):
            records.append({"gene": g, "cell_type": ct,
                            "mean_expr": lognorm[:, gi].mean(),
                            "pct_expr":  (block[:, gi] > 0).mean() * 100})
    return pd.DataFrame(records)

print("Helpers defined.")

## 7. EWCE Overview Heatmap

In [ ]:
df_ewce_all = pd.read_csv(f"{OUT}/ewce_results.csv")
pivot_ses   = df_ewce_all.pivot(index="disease", columns="cell_type", values="ses")
pivot_fdr   = df_ewce_all.pivot(index="disease", columns="cell_type", values="fdr")
ct_order    = VASC_TYPES + [c for c in pivot_ses.columns if c not in VASC_TYPES]
pivot_ses   = pivot_ses.reindex(columns=ct_order)
pivot_fdr   = pivot_fdr.reindex(columns=ct_order)
vasc_cols   = [c for c in VASC_TYPES if c in pivot_ses.columns]
pivot_ses["_vm"] = pivot_ses[vasc_cols].max(axis=1)
pivot_ses   = pivot_ses.sort_values("_vm", ascending=False).drop(columns="_vm")
pivot_fdr   = pivot_fdr.loc[pivot_ses.index]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot_ses, cmap="RdBu_r", center=0, vmin=-4, vmax=16,
            linewidths=0.5, linecolor="#cccccc",
            cbar_kws={"label": "SES", "shrink": 0.6}, ax=ax)
for ri, dis in enumerate(pivot_ses.index):
    for ci, ct in enumerate(ct_order):
        fv = pivot_fdr.loc[dis, ct] if ct in pivot_fdr.columns else np.nan
        if pd.notna(fv) and fv < 0.05:
            ax.text(ci+0.5, ri+0.5, "*", ha="center", va="center",
                    fontsize=14, color="black", fontweight="bold")
ax.axvline(len(vasc_cols), color="black", linewidth=2, linestyle="--")
ax.set_title("EWCE — All Diseases × All Cell Types  (* FDR < 0.05)", fontsize=11)
# ax.tick_params(axis='x', labelrotation=35)
plt.tight_layout()
fig.savefig(f"{OUT}/ewce_overview_all_celltypes.svg", bbox_inches="tight")
plt.show()
print("Saved: ewce_overview_all_celltypes.svg")

## 8. Dot Plot — Top EWCE Genes Across All Cell Classes

In [ ]:
ALL_CT    = obs["Cell_class"].cat.categories.tolist()
CT_ORDER  = VASC_TYPES + sorted([c for c in ALL_CT if c not in VASC_TYPES])

print("Computing dot plot stats ...")
all_top_genes = df_top["gene_name"].unique().tolist()
df_dot = compute_dotplot_stats(adata, obs, all_top_genes, CT_ORDER)
df_dot = df_dot.merge(
    df_top[["gene_name","disease","cell_type"]].rename(
        columns={"cell_type":"ewce_cell_type"}),
    left_on="gene", right_on="gene_name", how="left").drop(columns="gene_name")

gene_order = list(dict.fromkeys(
    df_top.sort_values(["cell_type","rank"])["gene_name"].tolist()))
print(f"Genes ({len(gene_order)}): {gene_order}")

In [ ]:
n_g, n_ct_dp = len(gene_order), len(CT_ORDER)
fig, ax = plt.subplots(figsize=(9, 4.1))

for _, row in df_dot.iterrows():
    if row["gene"] not in gene_order: continue
    xpos  = gene_order.index(row["gene"])
    ypos  = CT_ORDER.index(row["cell_type"])
    color = CT_COLORS.get(row["cell_type"], "#AAAAAA")
    size  = max(10, row["pct_expr"] * 4)
    ax.scatter(xpos, ypos, s=size, c=[color], alpha=0.85,
               edgecolors="k", linewidths=0.35, zorder=3)
    if row["mean_expr"] >= 0.3:
        ax.text(xpos, ypos, f'{row["mean_expr"]:.1f}', ha="center", va="center",
                fontsize=4.5, color="white", fontweight="bold")

ax.axhline(len(VASC_TYPES)-0.5, color="black", linewidth=1.5, linestyle="--", zorder=4)
group_boundaries = []
for ct in VASC_TYPES:
    ct_gs = df_top[df_top["cell_type"]==ct]["gene_name"].tolist()
    if ct_gs and ct_gs[-1] in gene_order:
        group_boundaries.append(gene_order.index(ct_gs[-1]) + 0.5)
for xb in group_boundaries[:-1]:
    ax.axvline(xb, color="#555555", linewidth=1.2, linestyle=":", zorder=4)

ax.set_xticks(range(n_g))
ax.set_xticklabels(gene_order, rotation=45, ha="right", fontsize=8, fontstyle="italic")
ax.set_yticks(range(n_ct_dp)); ax.set_yticklabels(CT_ORDER, fontsize=9)
ax.set_xlim(-0.5, n_g-0.5); ax.set_ylim(-0.5, n_ct_dp-0.5)
ax.grid(axis="x", alpha=0.2, linestyle="--")
ax.set_title("EWCE Top Disease Genes — All Cell Classes\n"
             "(size = % expressed; label = mean log-norm)", fontsize=10)

ax2 = ax.twiny(); ax2.set_xlim(ax.get_xlim()); ax2.set_xticks([])
for xi, g in enumerate(gene_order):
    dis = df_top[df_top["gene_name"]==g]["disease"].iloc[0]           if (df_top["gene_name"]==g).any() else "unknown"
    ax2.axvspan(xi-0.5, xi+0.5, color=DISEASE_COLORS.get(dis,"#AAAAAA"), alpha=0.25)
ax2.set_xlabel("Disease association (colour strip)", fontsize=8)

dis_patches  = [mpatches.Patch(color=c, label=d[:28], alpha=0.75)
                for d, c in DISEASE_COLORS.items() if d in df_top["disease"].values]
ct_patches   = [mpatches.Patch(color=CT_COLORS.get(c,"#AAA"), label=c) for c in CT_ORDER]
size_handles = [plt.scatter([],[],s=p*4,color="grey",alpha=0.7,
                            edgecolors="k",linewidths=0.5,label=f"{p}%") for p in [10,25,50,75]]
fig.legend(handles=dis_patches,  loc="lower left",   ncol=1, fontsize=7,
           bbox_to_anchor=(0.01,-0.26), title="Disease",    frameon=False)
fig.legend(handles=ct_patches,   loc="lower center", ncol=2, fontsize=7,
           bbox_to_anchor=(0.5, -0.26), title="Cell Class", frameon=False)
fig.legend(handles=size_handles, loc="lower right",  ncol=1, fontsize=7,
           bbox_to_anchor=(0.99,-0.26), title="% Expressed",frameon=False)
plt.tight_layout()
fig.savefig(f"{OUT}/ewce_dotplot_all_celltypes.svg", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: ewce_dotplot_all_celltypes.svg")

## 9. Pseudobulk Heatmaps Per Vascular Cell Type

In [ ]:
import seaborn as sns
pseudobulk_data = {}

for ct in VASC_TYPES:
    print(f"\n── {ct} ──")
    ct_genes_df  = df_top[df_top["cell_type"]==ct].sort_values("rank")
    genes        = ct_genes_df["gene_name"].tolist()
    if not genes: print("  No top genes."); continue

    df_z, df_logcpm = compute_pseudobulk(adata, obs, ct, genes)
    if df_z.empty: print(f"  No regions with ≥{MIN_CELLS} cells."); continue

    gene_meta    = ct_genes_df.set_index("gene_name")
    cols_ordered = [g for g in genes if g in df_z.columns]
    df_z_ord     = df_z[cols_ordered]

    csv_path = f"{OUT}/heatmap_pseudobulk_{ct}.csv"
    df_z_ord.to_csv(csv_path)
    pseudobulk_data[ct] = df_z_ord
    print(f"  Pseudobulk CSV: {csv_path}  ({df_z_ord.shape})")

    fig_h = max(8, len(df_z_ord)*0.28)
    fig_w = max(7, len(cols_ordered)*0.5+2.5)
    fig, axes = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(df_z_ord, cmap="RdBu_r", center=0, vmin=-2, vmax=2,
                linewidths=0.4, linecolor="#dddddd",
                cbar_kws={"label": "Z-score (log₂CPM)", "shrink": 0.6}, ax=axes)

    for tick in axes.get_xticklabels():
        gene = tick.get_text()
        dis  = gene_meta.loc[gene,"disease"] if gene in gene_meta.index else "unknown"
        tick.set_color(DISEASE_COLORS.get(dis, "#000000"))
        tick.set_fontsize(8)
    axes.set_xticklabels(axes.get_xticklabels(), rotation=45, ha="right")
    axes.set_yticklabels(axes.get_yticklabels(), fontsize=8, rotation=0)
    axes.set_title(f"{ct} — Top EWCE Disease Genes × Brain Region\n"
                   "(gene colour = disease; z-score pseudobulk log₂CPM)", fontsize=10, pad=14)

    ax_strip = axes.inset_axes([0,1.01,1,0.035], transform=axes.transAxes)
    ax_strip.set_xlim(0, len(cols_ordered)); ax_strip.set_ylim(0,1); ax_strip.axis("off")
    # for xi, g_name in enumerate(cols_ordered):
    #     pw = gene_meta.loc[g_name,"pathway_short"] if g_name in gene_meta.index else "Other"
    #     ax_strip.barh(0.5, 1, left=xi, height=1,
    #                   color=PATHWAY_COLORS.get(pw,"#AAAAAA"), edgecolor="white", linewidth=0.3)

    # pw_used  = gene_meta.loc[cols_ordered,"pathway_short"].unique() if cols_ordered else []
    dis_used = gene_meta.loc[cols_ordered,"disease"].unique()        if cols_ordered else []
    # fig.legend(handles=[mpatches.Patch(color=PATHWAY_COLORS.get(p,"#AAA"), label=p[:28])
    #                     for p in pw_used],
    #            loc="upper left", fontsize=6, bbox_to_anchor=(1.01,1.0),
    #            title="Pathway group", frameon=True)
    fig.legend(handles=[mpatches.Patch(color=DISEASE_COLORS.get(d,"#AAA"), label=d[:28], alpha=0.8)
                        for d in dis_used],
               loc="upper left", fontsize=6, bbox_to_anchor=(1.01,0.5),
               title="Disease", frameon=True)
    plt.tight_layout()
    svg_path = f"{OUT}/heatmap_{ct}.svg"
    fig.savefig(svg_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {svg_path}")

## 11. Summary

All outputs saved to `../Results/`. Figures displayed inline above.